> **Public copy.** The outputs below are from the original run on the private database. After that run, product, vendor and system names and release numbers in the code, text and tables were replaced with the public codes used everywhere in this repository (Vendor A to E; releases such as 1.51 and "Vendor C 3.0"), so every number is exactly as computed. The database itself is not published; the anonymised data files are in `data/`.


# 08 The label: which component-release rows are high risk?

**Goal:** turn "what happened" into a yes/no answer the model can learn: was this component **high risk** in this release?

**The business question.** Before a release starts, the QA lead wants to know **where to focus testing**: which components are likely to be defect hotspots. No incident or hotfix data exists, so the label uses **defects found in testing per 10 person-days** (agreed with Jagan as the best available measure). Working per person-day stops big releases looking risky just because more testing happened.

**Three options, compared side by side** (Jagan asked for all three, 24 Nov 2026). All need at least 2 defects, so one stray defect never makes a row high risk.

| Option | High risk when ... |
| --- | --- |
| A. Worse than its own normal | the component's defects per 10 person-days are in the top quarter of **its own** history in its vendor stream |
| D. Hotspot in its stream | the component's defects per 10 person-days are in the top quarter of **all components** in its vendor stream |
| D-sev. Hotspot, severity-weighted | as D, but defects are weighted by priority (Critical 8, High 4, Medium 2, Low 1) |

**How the lines are set:** "top quarter" means above the 75th percentile, taken from the **training releases with real tracker effort** only (not the test releases, and not 1.37 or Vendor C 3.0, whose effort is estimated). Vendor D has no training release, so it has no line and none of its rows can be high risk.

**The four tests, fixed before looking at the results:**
1. **Business fit:** does it answer "where should we focus testing in the next release?"
2. **Fair to release size:** it should mark about the same share of rows in big and small releases.
3. **Stable over time:** it should not change just because the way priority is rated changed ("Medium" only appears from 2022).
4. **Learnable:** can history rank its hotspots better than guessing? (Training releases only, in three time-ordered folds.)

In [1]:
import sqlite3, sys
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

# Works whether you run from the project folder or from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
CLEAN = ROOT / 'data' / 'private' / 'hindsight_clean.sqlite'  # our cleaned tables
pd.set_option('display.width', 200)
print('Reading from:', CLEAN)
with sqlite3.connect(CLEAN) as cc:
    fact = pd.read_sql('select * from fact_component_release', cc, parse_dates=['window_start', 'window_end'])
fact['effort_estimated'] = fact.effort_estimated.astype(bool)
basis = fact[(fact.split == 'train') & ~fact.effort_estimated]     # rows that set the lines
print(len(fact), 'rows;', len(basis), 'training rows with real effort set the lines')

Reading from: data/private/hindsight_clean.sqlite
541 rows; 357 training rows with real effort set the lines


## 1. The three labels

In [2]:
MIN_DEFECTS = 2
own_line = basis.groupby(['vendor', 'component']).density_per_10pd.quantile(0.75).rename('line_A')
stream_line = basis.groupby('vendor').density_per_10pd.quantile(0.75).rename('line_D')
sev_line = basis.groupby('vendor').sev_density_per_10pd.quantile(0.75).rename('line_Dsev')
fact = fact.join(own_line, on=['vendor', 'component']).join(stream_line, on='vendor').join(sev_line, on='vendor')

enough = fact.defects >= MIN_DEFECTS
fact['label_A'] = (enough & (fact.density_per_10pd > fact.line_A)).astype(int)
fact['label_D'] = (enough & (fact.density_per_10pd > fact.line_D)).astype(int)
fact['label_Dsev'] = (enough & (fact.sev_density_per_10pd > fact.line_Dsev)).astype(int)
LABELS = {'label_A': 'A. Worse than its own normal', 'label_D': 'D. Hotspot in its stream',
          'label_Dsev': 'D-sev. Hotspot, severity-weighted'}
print('Stream hotspot lines (per 10 person-days):')
pd.concat([stream_line, sev_line], axis=1).round(2)

Stream hotspot lines (per 10 person-days):


,line_D,line_Dsev
vendor,,
Vendor A,0.72,2.55
Vendor B,0.54,1.80
Vendor C,1.28,5.89
Vendor E,4.33,9.62


In [3]:
share = fact.groupby('split')[list(LABELS)].agg(['sum', 'mean']).T.unstack()
share.columns = [f'{s} {m}' for s, m in share.columns]
share.index = [LABELS[i] for i in share.index]
share.round(2)

,test sum,test mean,train sum,train mean
A. Worse than its own normal,18.0,0.11,84.0,0.22
D. Hotspot in its stream,24.0,0.15,96.0,0.25
"D-sev. Hotspot, severity-weighted",20.0,0.12,97.0,0.26


## 2. Test 2: fair to release size

The share of rows marked high risk in big releases (at or above the median person-days) and in small ones.

In [4]:
size = np.where(fact.person_days >= fact.person_days.median(), 'big releases', 'small releases')
fair = fact.groupby(size)[list(LABELS)].mean().T.round(2)
fair['gap'] = (fair['big releases'] - fair['small releases']).round(2)
fair.index = [LABELS[i] for i in fair.index]
fair

,big releases,small releases,gap
A. Worse than its own normal,0.20,0.18,0.02
D. Hotspot in its stream,0.23,0.21,0.02
"D-sev. Hotspot, severity-weighted",0.24,0.19,0.05


## 3. Test 3: stable over time

Training releases only: the share of rows marked high risk in releases that started **before 2022** and **from 2022**, when "Medium" priority appeared. A label that drops much more than the others is reacting to the rating change, not to quality.

In [5]:
tr = fact[fact.split == 'train']
era = np.where(tr.window_start < pd.Timestamp('2022-01-01'), 'started before 2022', 'started 2022-2024')
stable = tr.groupby(era)[list(LABELS)].mean().T.round(2)
stable['change'] = (stable['started 2022-2024'] - stable['started before 2022']).round(2)
stable.index = [LABELS[i] for i in stable.index]
stable

,started 2022-2024,started before 2022,change
A. Worse than its own normal,0.19,0.25,-0.06
D. Hotspot in its stream,0.25,0.26,-0.01
"D-sev. Hotspot, severity-weighted",0.24,0.28,-0.04


## 4. Test 4: learnable from history (training releases only)

Rank each fold's rows by the component's usual rate before the release (the plain usual rate for A and D, the severity-weighted usual rate for D-sev) and measure how well that ranking finds the high-risk rows. PR-AUC: 1 is perfect; guessing scores the share of high-risk rows. **Lift** = how many times better than guessing.

In [6]:
order = tr.groupby('release_key').window_start.min().sort_values()
n = len(order)
cuts = [int(n * q) for q in (0.55, 0.70, 0.85)] + [n]
folds = [order.index[cuts[i]:cuts[i + 1]] for i in range(3)]
score_of = {'label_A': 'hist_rate_per_10pd', 'label_D': 'hist_rate_per_10pd', 'label_Dsev': 'hist_sev_rate_per_10pd'}

rows = []
for lab, score in score_of.items():
    for i, val in enumerate(folds, start=1):
        v = tr[tr.release_key.isin(val)]
        if v[lab].sum() == 0:
            continue
        rows.append({'label': LABELS[lab], 'fold': i, 'guessing': v[lab].mean(),
                     'history ranking (PR-AUC)': average_precision_score(v[lab], v[score].fillna(0))})
learn = pd.DataFrame(rows).groupby('label')[['guessing', 'history ranking (PR-AUC)']].mean()
learn['lift'] = learn['history ranking (PR-AUC)'] / learn['guessing']
learn.round(2)

,guessing,history ranking (PR-AUC),lift
label,,,
A. Worse than its own normal,0.18,0.28,1.51
"D-sev. Hotspot, severity-weighted",0.24,0.68,2.87
D. Hotspot in its stream,0.24,0.69,2.90


## 5. How often the labels agree, and what they mark in the test releases

In [7]:
agree = pd.Series({'A and D agree': (fact.label_A == fact.label_D).mean(),
                   'D and D-sev agree': (fact.label_D == fact.label_Dsev).mean(),
                   'A and D-sev agree': (fact.label_A == fact.label_Dsev).mean()}).round(2)
print(agree.to_string(), end='\n\n')

def marked(lab):
    return fact[fact[lab] == 1].groupby('release_key').component.apply(lambda s: ', '.join(sorted(s)))
test_view = (fact[fact.split == 'test'].groupby(['release_key', 'vendor']).size().rename('rows').reset_index())
for lab in LABELS:
    test_view[lab.replace('label_', 'marked by ')] = test_view.release_key.map(marked(lab)).fillna('-')
test_view

A and D agree        0.83
D and D-sev agree    0.96
A and D-sev agree    0.82



,release_key,vendor,rows,marked by A,marked by D,marked by Dsev
0,1.51,Vendor A,17,-,-,-
1,1.52,Vendor A,17,"Payments and PED, Receipts and printing","Environment and infrastructure, Payments and P...","Environment and infrastructure, Payments and PED"
2,1.53,Vendor A,17,"Promotions, pricing and loyalty","Environment and infrastructure, Promotions, pr...","Environment and infrastructure, Promotions, pr..."
3,1.54,Vendor A,17,"Vendor D, Hardware and cash devices","Environment and infrastructure, Payments and P...","Environment and infrastructure, Payments and P..."
4,1.55,Vendor A,17,Environment and infrastructure,Environment and infrastructure,Environment and infrastructure
5,1.56,Vendor A,17,"Reporting, Vendor D, Hardware and...","Reporting, Vendor D, Self-checkou...","Reporting, Vendor D, Self-checkou..."
6,CI 11.1,Vendor B,10,Receipts and printing,"Receipts and printing, Self-checkout (SCO)","Receipts and printing, Self-checkout (SCO)"
7,CI 11.2,Vendor B,10,"Integrations and interfaces, Receipts and prin...","Integrations and interfaces, Receipts and prin...","Integrations and interfaces, Receipts and prin..."
8,CI 11.3,Vendor B,10,Environment and infrastructure,"Environment and infrastructure, Self-checkout ...","Environment and infrastructure, Self-checkout ..."
9,CI 11.4,Vendor B,10,-,Self-checkout (SCO),Self-checkout (SCO)


## 6. The choice: option D (agreed with Jagan, 24 Nov 2026)

**D. Hotspot in its stream** is best or joint best on all four tests:
1. **Business fit:** it answers "where should we focus testing in the next release?" Option A finds surprises instead.
2. **Fair:** it marks big and small releases almost equally (2-point gap; D-sev 5 points).
3. **Stable:** it barely changes when "Medium" priority appears in 2022 (1 point; D-sev 4, A 6).
4. **Learnable:** history ranks its hotspots about 2.9 times better than guessing (A about 1.5 times).

D-sev agrees with D on 96% of rows, so severity stays in the model as an input (the priority-weighted history), not in the label. Jagan checked D's hotspots in the test releases against his own memory of those releases and confirmed they look right. All three label columns are kept, so the choice can be changed with one setting.

In [8]:
CHOSEN = 'label_D'      # agreed with Jagan, 24 Nov 2026
fact['high_risk'] = fact[CHOSEN]
print('Chosen:', LABELS[CHOSEN])
print(fact.groupby('split').high_risk.agg(rows='size', high_risk='sum', share='mean').round(2))

Chosen: D. Hotspot in its stream
       rows  high_risk  share
split                        
test    161         24   0.15
train   380         96   0.25


## 7. Save

In [9]:
with sqlite3.connect(CLEAN) as cc:
    fact.to_sql('fact_labelled', cc, if_exists='replace', index=False)
    lines = (own_line.reset_index().merge(stream_line.reset_index(), on='vendor', how='left')
             .merge(sev_line.reset_index(), on='vendor', how='left'))
    lines.to_sql('label_thresholds', cc, if_exists='replace', index=False)
print('saved fact_labelled', fact.shape, 'and label_thresholds', lines.shape)

saved fact_labelled (541, 39) and label_thresholds (37, 5)


## What is next

Step 4: class weights versus SMOTE, to handle hotspots being the minority of rows. Then step 5, the model ladder.